In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

# =========================================================
# 1. USER SETTINGS
# =========================================================
BASE_DIR = Path(".")
PANEL_PATH = BASE_DIR / "Master_Modeling_Panel_v2.csv"

# Frozen overall winner from updated N3 (v4 plan)
XGB_PARAMS = {
    "n_estimators": 800,
    "max_depth": 3,
    "learning_rate": 0.01,
    "min_child_weight": 10,
    "subsample": 0.6,
    "colsample_bytree": 0.5,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": 0,
}

TARGET_COL = "targeted_in_year"
CATEGORICAL_FEATURE = "ff17_for_model"
ID_COLS = ["permno", "year"]

# =========================================================
# 2. LOAD PANEL
# =========================================================
df = pd.read_csv(PANEL_PATH, low_memory=False)
df[CATEGORICAL_FEATURE] = df["ff17_short"].fillna("Unclassified").astype(str)

# =========================================================
# 3. LOCKED RAW PREDICTOR SET
# =========================================================
continuous_features = [
    "roe",
    "assets_to_equity",
    "current_ratio",
    "ebitda",
    "ebitda_margin",
    "sales_to_assets",
    "sales_growth",
    "interest_coverage",
    "net_debt_to_ebitda",
    "fcf_to_capex",
    "market_cap",
    "ret_3m",
    "ret_6m",
    "ret_1y",
    "ret_2y",
    "ret_3y",
    "ret_5y",
    "volatility_30d",
    "volatility_90d",
    "volatility_180d",
    "turnover_30d",
    "dividend_payout_ratio",
    "buyback_yield",
    "price_to_book",
    "ev_to_sales",
    "ev_to_ebitda",
    "tobins_q",
    "fcf_yield",
    "prior_campaign_count_3y",
    "prior_settlement_count_3y",
    "prior_management_favorable_count_3y",
    "prior_dissident_favorable_count_3y",
    "prior_campaign_count_5y",
    "prior_settlement_count_5y",
    "prior_management_favorable_count_5y",
    "prior_dissident_favorable_count_5y",
    "ff49_other_permno_years_in_category",
    "ret_3m_rel_peer",
    "ret_6m_rel_peer",
    "ret_1y_rel_peer",
    "ret_2y_rel_peer",
    "ret_3y_rel_peer",
    "ret_5y_rel_peer",
    "log_ev_to_sales_rel_peer",
    "log_price_to_book_rel_peer",
    "log_tobins_q_rel_peer",
    "log_ev_to_ebitda_rel_peer",
    "ebitda_margin_rel_peer",
    "sales_growth_rel_peer",
    "roe_rel_peer",
    "board_size",
    "board_female_ratio",
    "ceo_tenure",
    "top10_inst_conc_within_13f",
    "inst_ownership_pct_13f",
]

binary_features = [
    "prior_activism_3y",
    "prior_activism_5y",
    "history_3y_observed",
    "history_5y_observed",
    "ff49_unclassified",
    "classified_board",
    "poison_pill",
    "dual_class",
    "iss_match_found",
    "rm_stale_gt_730",
    "board_match_found",
    "board_stale_gt_365",
    "ceo_is_female",
    "ceo_chair_duality",
    "ceo_match_found",
    "ceo_stale_gt_365",
    "inst_match_found",
]

all_predictors = continuous_features + binary_features + [CATEGORICAL_FEATURE]
required_cols = all_predictors + [TARGET_COL] + ID_COLS
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

# =========================================================
# 4. HELPERS
# =========================================================
FF17_CATEGORIES = [
    "Cars",
    "Chems",
    "Clths",
    "Cnstr",
    "Cnsum",
    "Durbl",
    "FabPr",
    "Finan",
    "Food",
    "Machn",
    "Mines",
    "Oil",
    "Other",
    "Rtail",
    "Steel",
    "Trans",
    "Utils",
    "Unclassified",
]

def evaluate_predictions(y_true: pd.Series, y_prob: np.ndarray) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    return {
        "pr_auc": average_precision_score(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "brier_score": brier_score_loss(y_true, y_prob),
    }

SUMMARY_CSV = BASE_DIR / "Model_R1_AlternativeSampleStart_v1_Stage_Metrics.csv"
CV_FOLD_METRICS_CSV = BASE_DIR / "Model_R1_AlternativeSampleStart_v1_SelectedHP_DevCV_Fold_Metrics.csv"
PREDICTIONS_CSV = BASE_DIR / "Model_R1_AlternativeSampleStart_v1_Predictions.csv"
SPEC_SUMMARY_CSV = BASE_DIR / "Model_R1_AlternativeSampleStart_v1_Spec_Summary.csv"

# =========================================================
# 5. ROBUSTNESS DESIGN: alternative sample start year
# =========================================================
ROBUSTNESS_ID = "R1"
ROBUSTNESS_DESCRIPTION = "Alternative sample start year: start in 2011 instead of 2010"

df = df[df["year"].between(2011, 2024)].copy()

def build_pipeline() -> Pipeline:
    continuous_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )

    binary_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unclassified")),
            (
                "onehot",
                OneHotEncoder(
                    categories=[FF17_CATEGORIES],
                    drop=["Other"],
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("cont", continuous_transformer, continuous_features),
            ("bin", binary_transformer, binary_features),
            ("ff17", categorical_transformer, [CATEGORICAL_FEATURE]),
        ],
        remainder="drop",
    )

    model = XGBClassifier(**XGB_PARAMS)

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

CV_FOLD_DEFS = [
    {"fold": "fold_1", "train_years": list(range(2011, 2016)), "val_years": [2016]},
    {"fold": "fold_2", "train_years": list(range(2011, 2017)), "val_years": [2017]},
    {"fold": "fold_3", "train_years": list(range(2011, 2018)), "val_years": [2018]},
    {"fold": "fold_4", "train_years": list(range(2011, 2019)), "val_years": [2019]},
    {"fold": "fold_5", "train_years": list(range(2011, 2020)), "val_years": [2020]},
    {"fold": "fold_6", "train_years": list(range(2011, 2021)), "val_years": [2021]},
]

run_start = time.time()
print(f"{ROBUSTNESS_ID} {ROBUSTNESS_DESCRIPTION}: starting six-fold evaluation...")

cv_rows = []
prediction_frames = []

for i, fold_def in enumerate(CV_FOLD_DEFS, start=1):
    train_years = fold_def["train_years"]
    val_years = fold_def["val_years"]

    train_df = df[df["year"].isin(train_years)].copy()
    val_df = df[df["year"].isin(val_years)].copy()

    X_train = train_df[all_predictors].copy()
    y_train = train_df[TARGET_COL].copy()
    X_val = val_df[all_predictors].copy()
    y_val = val_df[TARGET_COL].copy()

    print(
        f"[CV] Fold {i}/6 | train={min(train_years)}-{max(train_years)} | "
        f"val={min(val_years)} | train_rows={len(train_df)} | val_rows={len(val_df)}"
    )

    pipeline = build_pipeline()
    pipeline.fit(X_train, y_train)
    val_prob = pipeline.predict_proba(X_val)[:, 1]

    fold_metrics = evaluate_predictions(y_val, val_prob)
    fold_metrics.update(
        {
            "robustness_id": ROBUSTNESS_ID,
            "description": ROBUSTNESS_DESCRIPTION,
            "fold": fold_def["fold"],
            "train_year_start": min(train_years),
            "train_year_end": max(train_years),
            "val_year_start": min(val_years),
            "val_year_end": max(val_years),
            "train_rows": len(train_df),
            "train_positives": int(y_train.sum()),
            "train_positive_rate": float(y_train.mean()),
            "val_rows": len(val_df),
            "val_positives": int(y_val.sum()),
            "val_positive_rate": float(y_val.mean()),
            "mean_predicted_probability": float(np.mean(val_prob)),
        }
    )
    cv_rows.append(fold_metrics)

    prediction_frames.append(
        pd.DataFrame(
            {
                "robustness_id": ROBUSTNESS_ID,
                "description": ROBUSTNESS_DESCRIPTION,
                "permno": val_df["permno"].to_numpy(),
                "year": val_df["year"].to_numpy(),
                "evaluation_stage": "development_cv",
                "fold": fold_def["fold"],
                "actual": y_val.to_numpy(),
                "predicted_probability": val_prob,
            }
        )
    )

cv_df = pd.DataFrame(cv_rows)
cv_df.to_csv(CV_FOLD_METRICS_CSV, index=False)

dev_df = df[df["year"].between(2011, 2021)].copy()
test_df = df[df["year"].between(2022, 2024)].copy()

X_dev = dev_df[all_predictors].copy()
y_dev = dev_df[TARGET_COL].copy()
X_test = test_df[all_predictors].copy()
y_test = test_df[TARGET_COL].copy()

final_pipeline = build_pipeline()
final_pipeline.fit(X_dev, y_dev)
test_prob = final_pipeline.predict_proba(X_test)[:, 1]
test_metrics = evaluate_predictions(y_test, test_prob)

prediction_frames.append(
    pd.DataFrame(
        {
            "robustness_id": ROBUSTNESS_ID,
            "description": ROBUSTNESS_DESCRIPTION,
            "permno": test_df["permno"].to_numpy(),
            "year": test_df["year"].to_numpy(),
            "evaluation_stage": "final_test",
            "fold": "final_test",
            "actual": y_test.to_numpy(),
            "predicted_probability": test_prob,
        }
    )
)
predictions_df = pd.concat(prediction_frames, ignore_index=True)
predictions_df.to_csv(PREDICTIONS_CSV, index=False)

stage_df = pd.DataFrame(
    [
        {
            "stage": "development_cv_mean",
            "robustness_id": ROBUSTNESS_ID,
            "description": ROBUSTNESS_DESCRIPTION,
            "pr_auc": float(cv_df["pr_auc"].mean()),
            "roc_auc": float(cv_df["roc_auc"].mean()),
            "brier_score": float(cv_df["brier_score"].mean()),
            "rows": np.nan,
            "positives": np.nan,
            "positive_rate": np.nan,
            "mean_predicted_probability": np.nan,
            "sample_start_year": 2011,
            "sample_end_year": 2024,
            "test_window": "2022-2024",
        },
        {
            "stage": "development_cv_std",
            "robustness_id": ROBUSTNESS_ID,
            "description": ROBUSTNESS_DESCRIPTION,
            "pr_auc": float(cv_df["pr_auc"].std(ddof=1)),
            "roc_auc": float(cv_df["roc_auc"].std(ddof=1)),
            "brier_score": float(cv_df["brier_score"].std(ddof=1)),
            "rows": np.nan,
            "positives": np.nan,
            "positive_rate": np.nan,
            "mean_predicted_probability": np.nan,
            "sample_start_year": 2011,
            "sample_end_year": 2024,
            "test_window": "2022-2024",
        },
        {
            "stage": "final_test",
            "robustness_id": ROBUSTNESS_ID,
            "description": ROBUSTNESS_DESCRIPTION,
            "pr_auc": float(test_metrics["pr_auc"]),
            "roc_auc": float(test_metrics["roc_auc"]),
            "brier_score": float(test_metrics["brier_score"]),
            "rows": int(len(test_df)),
            "positives": int(y_test.sum()),
            "positive_rate": float(y_test.mean()),
            "mean_predicted_probability": float(np.mean(test_prob)),
            "sample_start_year": 2011,
            "sample_end_year": 2024,
            "test_window": "2022-2024",
        },
    ]
)
stage_df.to_csv(SUMMARY_CSV, index=False)

spec_df = pd.DataFrame(
    [
        {
            "robustness_id": ROBUSTNESS_ID,
            "description": ROBUSTNESS_DESCRIPTION,
            "frozen_model_variant": "updated_N3_no_winsor",
            "preprocessing_variant": "median_imputation_only",
            "imbalance_choice": "none",
            "sample_start_year": 2011,
            "development_window": "2011-2021",
            "test_window": "2022-2024",
            "target_col": TARGET_COL,
            "predictor_count": len(all_predictors),
            "notes": "Alternative sample start year robustness relative to main 2010-start specification.",
        }
    ]
)
spec_df.to_csv(SPEC_SUMMARY_CSV, index=False)

print(f"{ROBUSTNESS_ID} complete in {(time.time() - run_start)/60.0:.2f} minutes.")

R1 Alternative sample start year: start in 2011 instead of 2010: starting six-fold evaluation...
[CV] Fold 1/6 | train=2011-2015 | val=2016 | train_rows=19804 | val_rows=3888
[CV] Fold 2/6 | train=2011-2016 | val=2017 | train_rows=23692 | val_rows=3811
[CV] Fold 3/6 | train=2011-2017 | val=2018 | train_rows=27503 | val_rows=3794
[CV] Fold 4/6 | train=2011-2018 | val=2019 | train_rows=31297 | val_rows=3805
[CV] Fold 5/6 | train=2011-2019 | val=2020 | train_rows=35102 | val_rows=3907
[CV] Fold 6/6 | train=2011-2020 | val=2021 | train_rows=39009 | val_rows=4535
R1 complete in 0.29 minutes.


In [2]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

# =========================================================
# 1. USER SETTINGS
# =========================================================
BASE_DIR = Path(".")
PANEL_PATH = BASE_DIR / "Master_Modeling_Panel_v2.csv"

# Frozen overall winner from updated N3 (v4 plan)
XGB_PARAMS = {
    "n_estimators": 800,
    "max_depth": 3,
    "learning_rate": 0.01,
    "min_child_weight": 10,
    "subsample": 0.6,
    "colsample_bytree": 0.5,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": 0,
}

TARGET_COL = "targeted_in_year"
CATEGORICAL_FEATURE = "ff17_for_model"
ID_COLS = ["permno", "year"]

# =========================================================
# 2. LOAD PANEL
# =========================================================
df = pd.read_csv(PANEL_PATH, low_memory=False)
df[CATEGORICAL_FEATURE] = df["ff17_short"].fillna("Unclassified").astype(str)

# =========================================================
# 3. LOCKED RAW PREDICTOR SET
# =========================================================
continuous_features = [
    "roe",
    "assets_to_equity",
    "current_ratio",
    "ebitda",
    "ebitda_margin",
    "sales_to_assets",
    "sales_growth",
    "interest_coverage",
    "net_debt_to_ebitda",
    "fcf_to_capex",
    "market_cap",
    "ret_3m",
    "ret_6m",
    "ret_1y",
    "ret_2y",
    "ret_3y",
    "ret_5y",
    "volatility_30d",
    "volatility_90d",
    "volatility_180d",
    "turnover_30d",
    "dividend_payout_ratio",
    "buyback_yield",
    "price_to_book",
    "ev_to_sales",
    "ev_to_ebitda",
    "tobins_q",
    "fcf_yield",
    "prior_campaign_count_3y",
    "prior_settlement_count_3y",
    "prior_management_favorable_count_3y",
    "prior_dissident_favorable_count_3y",
    "prior_campaign_count_5y",
    "prior_settlement_count_5y",
    "prior_management_favorable_count_5y",
    "prior_dissident_favorable_count_5y",
    "ff49_other_permno_years_in_category",
    "ret_3m_rel_peer",
    "ret_6m_rel_peer",
    "ret_1y_rel_peer",
    "ret_2y_rel_peer",
    "ret_3y_rel_peer",
    "ret_5y_rel_peer",
    "log_ev_to_sales_rel_peer",
    "log_price_to_book_rel_peer",
    "log_tobins_q_rel_peer",
    "log_ev_to_ebitda_rel_peer",
    "ebitda_margin_rel_peer",
    "sales_growth_rel_peer",
    "roe_rel_peer",
    "board_size",
    "board_female_ratio",
    "ceo_tenure",
    "top10_inst_conc_within_13f",
    "inst_ownership_pct_13f",
]

binary_features = [
    "prior_activism_3y",
    "prior_activism_5y",
    "history_3y_observed",
    "history_5y_observed",
    "ff49_unclassified",
    "classified_board",
    "poison_pill",
    "dual_class",
    "iss_match_found",
    "rm_stale_gt_730",
    "board_match_found",
    "board_stale_gt_365",
    "ceo_is_female",
    "ceo_chair_duality",
    "ceo_match_found",
    "ceo_stale_gt_365",
    "inst_match_found",
]

all_predictors = continuous_features + binary_features + [CATEGORICAL_FEATURE]
required_cols = all_predictors + [TARGET_COL] + ID_COLS
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

# =========================================================
# 4. HELPERS
# =========================================================
FF17_CATEGORIES = [
    "Cars",
    "Chems",
    "Clths",
    "Cnstr",
    "Cnsum",
    "Durbl",
    "FabPr",
    "Finan",
    "Food",
    "Machn",
    "Mines",
    "Oil",
    "Other",
    "Rtail",
    "Steel",
    "Trans",
    "Utils",
    "Unclassified",
]

def evaluate_predictions(y_true: pd.Series, y_prob: np.ndarray) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    return {
        "pr_auc": average_precision_score(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "brier_score": brier_score_loss(y_true, y_prob),
    }

SUMMARY_CSV = BASE_DIR / "Model_R2_AlternativePreprocessing_v1_Stage_Metrics.csv"
CV_FOLD_METRICS_CSV = BASE_DIR / "Model_R2_AlternativePreprocessing_v1_SelectedHP_DevCV_Fold_Metrics.csv"
PREDICTIONS_CSV = BASE_DIR / "Model_R2_AlternativePreprocessing_v1_Predictions.csv"
SPEC_SUMMARY_CSV = BASE_DIR / "Model_R2_AlternativePreprocessing_v1_Spec_Summary.csv"

# =========================================================
# 5. ROBUSTNESS DESIGN: alternative preprocessing / imputation
# =========================================================
ROBUSTNESS_ID = "R2"
ROBUSTNESS_DESCRIPTION = "Alternative preprocessing: mean imputation for continuous features"

df = df[df["year"].between(2010, 2024)].copy()

def build_pipeline() -> Pipeline:
    continuous_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="mean")),
        ]
    )

    binary_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unclassified")),
            (
                "onehot",
                OneHotEncoder(
                    categories=[FF17_CATEGORIES],
                    drop=["Other"],
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("cont", continuous_transformer, continuous_features),
            ("bin", binary_transformer, binary_features),
            ("ff17", categorical_transformer, [CATEGORICAL_FEATURE]),
        ],
        remainder="drop",
    )

    model = XGBClassifier(**XGB_PARAMS)

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

CV_FOLD_DEFS = [
    {"fold": "fold_1", "train_years": list(range(2010, 2016)), "val_years": [2016]},
    {"fold": "fold_2", "train_years": list(range(2010, 2017)), "val_years": [2017]},
    {"fold": "fold_3", "train_years": list(range(2010, 2018)), "val_years": [2018]},
    {"fold": "fold_4", "train_years": list(range(2010, 2019)), "val_years": [2019]},
    {"fold": "fold_5", "train_years": list(range(2010, 2020)), "val_years": [2020]},
    {"fold": "fold_6", "train_years": list(range(2010, 2021)), "val_years": [2021]},
]

run_start = time.time()
print(f"{ROBUSTNESS_ID} {ROBUSTNESS_DESCRIPTION}: starting six-fold evaluation...")

cv_rows = []
prediction_frames = []

for i, fold_def in enumerate(CV_FOLD_DEFS, start=1):
    train_years = fold_def["train_years"]
    val_years = fold_def["val_years"]

    train_df = df[df["year"].isin(train_years)].copy()
    val_df = df[df["year"].isin(val_years)].copy()

    X_train = train_df[all_predictors].copy()
    y_train = train_df[TARGET_COL].copy()
    X_val = val_df[all_predictors].copy()
    y_val = val_df[TARGET_COL].copy()

    print(
        f"[CV] Fold {i}/6 | train={min(train_years)}-{max(train_years)} | "
        f"val={min(val_years)} | train_rows={len(train_df)} | val_rows={len(val_df)}"
    )

    pipeline = build_pipeline()
    pipeline.fit(X_train, y_train)
    val_prob = pipeline.predict_proba(X_val)[:, 1]

    fold_metrics = evaluate_predictions(y_val, val_prob)
    fold_metrics.update(
        {
            "robustness_id": ROBUSTNESS_ID,
            "description": ROBUSTNESS_DESCRIPTION,
            "fold": fold_def["fold"],
            "train_year_start": min(train_years),
            "train_year_end": max(train_years),
            "val_year_start": min(val_years),
            "val_year_end": max(val_years),
            "train_rows": len(train_df),
            "train_positives": int(y_train.sum()),
            "train_positive_rate": float(y_train.mean()),
            "val_rows": len(val_df),
            "val_positives": int(y_val.sum()),
            "val_positive_rate": float(y_val.mean()),
            "mean_predicted_probability": float(np.mean(val_prob)),
        }
    )
    cv_rows.append(fold_metrics)

    prediction_frames.append(
        pd.DataFrame(
            {
                "robustness_id": ROBUSTNESS_ID,
                "description": ROBUSTNESS_DESCRIPTION,
                "permno": val_df["permno"].to_numpy(),
                "year": val_df["year"].to_numpy(),
                "evaluation_stage": "development_cv",
                "fold": fold_def["fold"],
                "actual": y_val.to_numpy(),
                "predicted_probability": val_prob,
            }
        )
    )

cv_df = pd.DataFrame(cv_rows)
cv_df.to_csv(CV_FOLD_METRICS_CSV, index=False)

dev_df = df[df["year"].between(2010, 2021)].copy()
test_df = df[df["year"].between(2022, 2024)].copy()

X_dev = dev_df[all_predictors].copy()
y_dev = dev_df[TARGET_COL].copy()
X_test = test_df[all_predictors].copy()
y_test = test_df[TARGET_COL].copy()

final_pipeline = build_pipeline()
final_pipeline.fit(X_dev, y_dev)
test_prob = final_pipeline.predict_proba(X_test)[:, 1]
test_metrics = evaluate_predictions(y_test, test_prob)

prediction_frames.append(
    pd.DataFrame(
        {
            "robustness_id": ROBUSTNESS_ID,
            "description": ROBUSTNESS_DESCRIPTION,
            "permno": test_df["permno"].to_numpy(),
            "year": test_df["year"].to_numpy(),
            "evaluation_stage": "final_test",
            "fold": "final_test",
            "actual": y_test.to_numpy(),
            "predicted_probability": test_prob,
        }
    )
)
predictions_df = pd.concat(prediction_frames, ignore_index=True)
predictions_df.to_csv(PREDICTIONS_CSV, index=False)

stage_df = pd.DataFrame(
    [
        {
            "stage": "development_cv_mean",
            "robustness_id": ROBUSTNESS_ID,
            "description": ROBUSTNESS_DESCRIPTION,
            "pr_auc": float(cv_df["pr_auc"].mean()),
            "roc_auc": float(cv_df["roc_auc"].mean()),
            "brier_score": float(cv_df["brier_score"].mean()),
            "rows": np.nan,
            "positives": np.nan,
            "positive_rate": np.nan,
            "mean_predicted_probability": np.nan,
            "sample_start_year": 2010,
            "sample_end_year": 2024,
            "test_window": "2022-2024",
        },
        {
            "stage": "development_cv_std",
            "robustness_id": ROBUSTNESS_ID,
            "description": ROBUSTNESS_DESCRIPTION,
            "pr_auc": float(cv_df["pr_auc"].std(ddof=1)),
            "roc_auc": float(cv_df["roc_auc"].std(ddof=1)),
            "brier_score": float(cv_df["brier_score"].std(ddof=1)),
            "rows": np.nan,
            "positives": np.nan,
            "positive_rate": np.nan,
            "mean_predicted_probability": np.nan,
            "sample_start_year": 2010,
            "sample_end_year": 2024,
            "test_window": "2022-2024",
        },
        {
            "stage": "final_test",
            "robustness_id": ROBUSTNESS_ID,
            "description": ROBUSTNESS_DESCRIPTION,
            "pr_auc": float(test_metrics["pr_auc"]),
            "roc_auc": float(test_metrics["roc_auc"]),
            "brier_score": float(test_metrics["brier_score"]),
            "rows": int(len(test_df)),
            "positives": int(y_test.sum()),
            "positive_rate": float(y_test.mean()),
            "mean_predicted_probability": float(np.mean(test_prob)),
            "sample_start_year": 2010,
            "sample_end_year": 2024,
            "test_window": "2022-2024",
        },
    ]
)
stage_df.to_csv(SUMMARY_CSV, index=False)

spec_df = pd.DataFrame(
    [
        {
            "robustness_id": ROBUSTNESS_ID,
            "description": ROBUSTNESS_DESCRIPTION,
            "frozen_model_variant": "updated_N3_no_winsor",
            "preprocessing_variant": "mean_imputation_only",
            "imbalance_choice": "none",
            "development_window": "2010-2021",
            "test_window": "2022-2024",
            "target_col": TARGET_COL,
            "predictor_count": len(all_predictors),
            "notes": "Narrow preprocessing robustness: continuous mean imputation replaces median imputation.",
        }
    ]
)
spec_df.to_csv(SPEC_SUMMARY_CSV, index=False)

print(f"{ROBUSTNESS_ID} complete in {(time.time() - run_start)/60.0:.2f} minutes.")

R2 Alternative preprocessing: mean imputation for continuous features: starting six-fold evaluation...
[CV] Fold 1/6 | train=2010-2015 | val=2016 | train_rows=24053 | val_rows=3888
[CV] Fold 2/6 | train=2010-2016 | val=2017 | train_rows=27941 | val_rows=3811
[CV] Fold 3/6 | train=2010-2017 | val=2018 | train_rows=31752 | val_rows=3794
[CV] Fold 4/6 | train=2010-2018 | val=2019 | train_rows=35546 | val_rows=3805
[CV] Fold 5/6 | train=2010-2019 | val=2020 | train_rows=39351 | val_rows=3907
[CV] Fold 6/6 | train=2010-2020 | val=2021 | train_rows=43258 | val_rows=4535
R2 complete in 0.31 minutes.


In [3]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

# =========================================================
# 1. USER SETTINGS
# =========================================================
BASE_DIR = Path(".")
PANEL_PATH = BASE_DIR / "Master_Modeling_Panel_v2.csv"

# Frozen overall winner from updated N3 (v4 plan)
XGB_PARAMS = {
    "n_estimators": 800,
    "max_depth": 3,
    "learning_rate": 0.01,
    "min_child_weight": 10,
    "subsample": 0.6,
    "colsample_bytree": 0.5,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": 0,
}

TARGET_COL = "targeted_in_year"
CATEGORICAL_FEATURE = "ff17_for_model"
ID_COLS = ["permno", "year"]

# =========================================================
# 2. LOAD PANEL
# =========================================================
df = pd.read_csv(PANEL_PATH, low_memory=False)
df[CATEGORICAL_FEATURE] = df["ff17_short"].fillna("Unclassified").astype(str)

# =========================================================
# 3. LOCKED RAW PREDICTOR SET
# =========================================================
continuous_features = [
    "roe",
    "assets_to_equity",
    "current_ratio",
    "ebitda",
    "ebitda_margin",
    "sales_to_assets",
    "sales_growth",
    "interest_coverage",
    "net_debt_to_ebitda",
    "fcf_to_capex",
    "market_cap",
    "ret_3m",
    "ret_6m",
    "ret_1y",
    "ret_2y",
    "ret_3y",
    "ret_5y",
    "volatility_30d",
    "volatility_90d",
    "volatility_180d",
    "turnover_30d",
    "dividend_payout_ratio",
    "buyback_yield",
    "price_to_book",
    "ev_to_sales",
    "ev_to_ebitda",
    "tobins_q",
    "fcf_yield",
    "prior_campaign_count_3y",
    "prior_settlement_count_3y",
    "prior_management_favorable_count_3y",
    "prior_dissident_favorable_count_3y",
    "prior_campaign_count_5y",
    "prior_settlement_count_5y",
    "prior_management_favorable_count_5y",
    "prior_dissident_favorable_count_5y",
    "ff49_other_permno_years_in_category",
    "ret_3m_rel_peer",
    "ret_6m_rel_peer",
    "ret_1y_rel_peer",
    "ret_2y_rel_peer",
    "ret_3y_rel_peer",
    "ret_5y_rel_peer",
    "log_ev_to_sales_rel_peer",
    "log_price_to_book_rel_peer",
    "log_tobins_q_rel_peer",
    "log_ev_to_ebitda_rel_peer",
    "ebitda_margin_rel_peer",
    "sales_growth_rel_peer",
    "roe_rel_peer",
    "board_size",
    "board_female_ratio",
    "ceo_tenure",
    "top10_inst_conc_within_13f",
    "inst_ownership_pct_13f",
]

binary_features = [
    "prior_activism_3y",
    "prior_activism_5y",
    "history_3y_observed",
    "history_5y_observed",
    "ff49_unclassified",
    "classified_board",
    "poison_pill",
    "dual_class",
    "iss_match_found",
    "rm_stale_gt_730",
    "board_match_found",
    "board_stale_gt_365",
    "ceo_is_female",
    "ceo_chair_duality",
    "ceo_match_found",
    "ceo_stale_gt_365",
    "inst_match_found",
]

all_predictors = continuous_features + binary_features + [CATEGORICAL_FEATURE]
required_cols = all_predictors + [TARGET_COL] + ID_COLS
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

# =========================================================
# 4. HELPERS
# =========================================================
FF17_CATEGORIES = [
    "Cars",
    "Chems",
    "Clths",
    "Cnstr",
    "Cnsum",
    "Durbl",
    "FabPr",
    "Finan",
    "Food",
    "Machn",
    "Mines",
    "Oil",
    "Other",
    "Rtail",
    "Steel",
    "Trans",
    "Utils",
    "Unclassified",
]

def evaluate_predictions(y_true: pd.Series, y_prob: np.ndarray) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    return {
        "pr_auc": average_precision_score(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "brier_score": brier_score_loss(y_true, y_prob),
    }

SUMMARY_CSV = BASE_DIR / "Model_R3_NextYearLabel_v1_Stage_Metrics.csv"
CV_FOLD_METRICS_CSV = BASE_DIR / "Model_R3_NextYearLabel_v1_SelectedHP_DevCV_Fold_Metrics.csv"
PREDICTIONS_CSV = BASE_DIR / "Model_R3_NextYearLabel_v1_Predictions.csv"
SPEC_SUMMARY_CSV = BASE_DIR / "Model_R3_NextYearLabel_v1_Spec_Summary.csv"

# =========================================================
# 5. ROBUSTNESS DESIGN: next-year label
# =========================================================
ROBUSTNESS_ID = "R3"
ROBUSTNESS_DESCRIPTION = "Next-year label robustness using targeted_next_year"

# Build targeted_next_year within permno
df = df[df["year"].between(2010, 2024)].copy()
df = df.sort_values(["permno", "year"]).copy()
df["targeted_next_year"] = df.groupby("permno")[TARGET_COL].shift(-1)

# Restrict to years where next-year outcome is observable
df = df[df["year"].between(2010, 2023)].copy()
df = df[df["targeted_next_year"].notna()].copy()
TARGET_COL = "targeted_next_year"

def build_pipeline() -> Pipeline:
    continuous_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )

    binary_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unclassified")),
            (
                "onehot",
                OneHotEncoder(
                    categories=[FF17_CATEGORIES],
                    drop=["Other"],
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("cont", continuous_transformer, continuous_features),
            ("bin", binary_transformer, binary_features),
            ("ff17", categorical_transformer, [CATEGORICAL_FEATURE]),
        ],
        remainder="drop",
    )

    model = XGBClassifier(**XGB_PARAMS)

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

CV_FOLD_DEFS = [
    {"fold": "fold_1", "train_years": list(range(2010, 2016)), "val_years": [2016]},
    {"fold": "fold_2", "train_years": list(range(2010, 2017)), "val_years": [2017]},
    {"fold": "fold_3", "train_years": list(range(2010, 2018)), "val_years": [2018]},
    {"fold": "fold_4", "train_years": list(range(2010, 2019)), "val_years": [2019]},
    {"fold": "fold_5", "train_years": list(range(2010, 2020)), "val_years": [2020]},
    {"fold": "fold_6", "train_years": list(range(2010, 2021)), "val_years": [2021]},
]

run_start = time.time()
print(f"{ROBUSTNESS_ID} {ROBUSTNESS_DESCRIPTION}: starting six-fold evaluation...")

cv_rows = []
prediction_frames = []

for i, fold_def in enumerate(CV_FOLD_DEFS, start=1):
    train_years = fold_def["train_years"]
    val_years = fold_def["val_years"]

    train_df = df[df["year"].isin(train_years)].copy()
    val_df = df[df["year"].isin(val_years)].copy()

    X_train = train_df[all_predictors].copy()
    y_train = train_df[TARGET_COL].copy().astype(int)
    X_val = val_df[all_predictors].copy()
    y_val = val_df[TARGET_COL].copy().astype(int)

    print(
        f"[CV] Fold {i}/6 | train={min(train_years)}-{max(train_years)} | "
        f"val={min(val_years)} | train_rows={len(train_df)} | val_rows={len(val_df)}"
    )

    pipeline = build_pipeline()
    pipeline.fit(X_train, y_train)
    val_prob = pipeline.predict_proba(X_val)[:, 1]

    fold_metrics = evaluate_predictions(y_val, val_prob)
    fold_metrics.update(
        {
            "robustness_id": ROBUSTNESS_ID,
            "description": ROBUSTNESS_DESCRIPTION,
            "fold": fold_def["fold"],
            "train_year_start": min(train_years),
            "train_year_end": max(train_years),
            "val_year_start": min(val_years),
            "val_year_end": max(val_years),
            "train_rows": len(train_df),
            "train_positives": int(y_train.sum()),
            "train_positive_rate": float(y_train.mean()),
            "val_rows": len(val_df),
            "val_positives": int(y_val.sum()),
            "val_positive_rate": float(y_val.mean()),
            "mean_predicted_probability": float(np.mean(val_prob)),
        }
    )
    cv_rows.append(fold_metrics)

    prediction_frames.append(
        pd.DataFrame(
            {
                "robustness_id": ROBUSTNESS_ID,
                "description": ROBUSTNESS_DESCRIPTION,
                "permno": val_df["permno"].to_numpy(),
                "year": val_df["year"].to_numpy(),
                "evaluation_stage": "development_cv",
                "fold": fold_def["fold"],
                "actual": y_val.to_numpy(),
                "predicted_probability": val_prob,
            }
        )
    )

cv_df = pd.DataFrame(cv_rows)
cv_df.to_csv(CV_FOLD_METRICS_CSV, index=False)

dev_df = df[df["year"].between(2010, 2021)].copy()
test_df = df[df["year"].between(2022, 2023)].copy()

X_dev = dev_df[all_predictors].copy()
y_dev = dev_df[TARGET_COL].copy().astype(int)
X_test = test_df[all_predictors].copy()
y_test = test_df[TARGET_COL].copy().astype(int)

final_pipeline = build_pipeline()
final_pipeline.fit(X_dev, y_dev)
test_prob = final_pipeline.predict_proba(X_test)[:, 1]
test_metrics = evaluate_predictions(y_test, test_prob)

prediction_frames.append(
    pd.DataFrame(
        {
            "robustness_id": ROBUSTNESS_ID,
            "description": ROBUSTNESS_DESCRIPTION,
            "permno": test_df["permno"].to_numpy(),
            "year": test_df["year"].to_numpy(),
            "evaluation_stage": "final_test",
            "fold": "final_test",
            "actual": y_test.to_numpy(),
            "predicted_probability": test_prob,
        }
    )
)
predictions_df = pd.concat(prediction_frames, ignore_index=True)
predictions_df.to_csv(PREDICTIONS_CSV, index=False)

stage_df = pd.DataFrame(
    [
        {
            "stage": "development_cv_mean",
            "robustness_id": ROBUSTNESS_ID,
            "description": ROBUSTNESS_DESCRIPTION,
            "pr_auc": float(cv_df["pr_auc"].mean()),
            "roc_auc": float(cv_df["roc_auc"].mean()),
            "brier_score": float(cv_df["brier_score"].mean()),
            "rows": np.nan,
            "positives": np.nan,
            "positive_rate": np.nan,
            "mean_predicted_probability": np.nan,
            "sample_start_year": 2010,
            "sample_end_year": 2023,
            "test_window": "2022-2023",
        },
        {
            "stage": "development_cv_std",
            "robustness_id": ROBUSTNESS_ID,
            "description": ROBUSTNESS_DESCRIPTION,
            "pr_auc": float(cv_df["pr_auc"].std(ddof=1)),
            "roc_auc": float(cv_df["roc_auc"].std(ddof=1)),
            "brier_score": float(cv_df["brier_score"].std(ddof=1)),
            "rows": np.nan,
            "positives": np.nan,
            "positive_rate": np.nan,
            "mean_predicted_probability": np.nan,
            "sample_start_year": 2010,
            "sample_end_year": 2023,
            "test_window": "2022-2023",
        },
        {
            "stage": "final_test",
            "robustness_id": ROBUSTNESS_ID,
            "description": ROBUSTNESS_DESCRIPTION,
            "pr_auc": float(test_metrics["pr_auc"]),
            "roc_auc": float(test_metrics["roc_auc"]),
            "brier_score": float(test_metrics["brier_score"]),
            "rows": int(len(test_df)),
            "positives": int(y_test.sum()),
            "positive_rate": float(y_test.mean()),
            "mean_predicted_probability": float(np.mean(test_prob)),
            "sample_start_year": 2010,
            "sample_end_year": 2023,
            "test_window": "2022-2023",
        },
    ]
)
stage_df.to_csv(SUMMARY_CSV, index=False)

spec_df = pd.DataFrame(
    [
        {
            "robustness_id": ROBUSTNESS_ID,
            "description": ROBUSTNESS_DESCRIPTION,
            "frozen_model_variant": "updated_N3_no_winsor",
            "preprocessing_variant": "median_imputation_only",
            "imbalance_choice": "none",
            "development_window": "2010-2021",
            "test_window": "2022-2023",
            "target_col": TARGET_COL,
            "predictor_count": len(all_predictors),
            "notes": "Next-year target robustness; 2024 excluded because 2025 outcome is unavailable.",
        }
    ]
)
spec_df.to_csv(SPEC_SUMMARY_CSV, index=False)

print(f"{ROBUSTNESS_ID} complete in {(time.time() - run_start)/60.0:.2f} minutes.")

R3 Next-year label robustness using targeted_next_year: starting six-fold evaluation...
[CV] Fold 1/6 | train=2010-2015 | val=2016 | train_rows=22490 | val_rows=3600
[CV] Fold 2/6 | train=2010-2016 | val=2017 | train_rows=26090 | val_rows=3561
[CV] Fold 3/6 | train=2010-2017 | val=2018 | train_rows=29651 | val_rows=3589
[CV] Fold 4/6 | train=2010-2018 | val=2019 | train_rows=33240 | val_rows=3577
[CV] Fold 5/6 | train=2010-2019 | val=2020 | train_rows=36817 | val_rows=3728
[CV] Fold 6/6 | train=2010-2020 | val=2021 | train_rows=40545 | val_rows=4344
R3 complete in 0.25 minutes.
